# 01 Preprocessing and Splits

This notebook runs the shared WESAD preprocessing pipeline from `src.preprocessing`. It creates the fixed subject splits, raw sequence windows, normalized sequence windows, labels, metadata, and preprocessing artifacts used by all model notebooks.


## 1. Imports and configuration


In [10]:
from pathlib import Path
import sys
import torch  # Load PyTorch before scientific stack on Windows.

CURRENT_DIRECTORY = Path.cwd().resolve()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from src.config import (
    PROJECT_ROOT as PACKAGE_PROJECT_ROOT,
    TRAIN_SUBJECTS,
    VALIDATION_SUBJECTS,
    TEST_SUBJECTS,
    SPLIT_SUBJECTS,
    SEQUENCE_CHANNELS,
    TARGET_HZ,
    WINDOW_SECONDS,
    WINDOW_SAMPLES,
    STRIDE_SECONDS,
    STRIDE_SAMPLES,
    RANDOM_SEED,
)

from src.preprocessing import run_preprocessing


Project root: C:\Users\User\Documents\deep-learning\deep-learning


## 2. Shared preprocessing implementation


In [11]:
# Preprocessing logic is implemented in src.preprocessing.run_preprocessing.


## 3. Fixed subject split and window configuration


In [12]:
SPLIT_SUBJECTS, TARGET_HZ, WINDOW_SECONDS, WINDOW_SAMPLES, STRIDE_SECONDS, STRIDE_SAMPLES, SEQUENCE_CHANNELS


({'train': ['S3', 'S4', 'S6', 'S7', 'S8', 'S9', 'S10', 'S13', 'S16', 'S17'],
  'validation': ['S5', 'S15'],
  'test': ['S2', 'S11', 'S14']},
 32,
 30,
 960,
 15,
 480,
 ['BVP', 'EDA', 'TEMP', 'ACC_x', 'ACC_y', 'ACC_z'])

## 4. Load each subject separately


`run_preprocessing` loads each subject pickle independently from the fixed split.


## 5. Extract BVP, EDA, TEMP and ACC


The notebook uses wrist BVP, EDA, TEMP, and 3-axis ACC only.


## 6. Align all signals to 32 Hz


BVP is downsampled from 64 Hz to 32 Hz with polyphase resampling. EDA and TEMP are linearly interpolated from 4 Hz, ACC remains at 32 Hz, and labels are sampled from the 700 Hz label stream.


## 7. Keep labels 1, 2 and 3


Only baseline, stress, and amusement are retained.


## 8. Create continuous condition segments


Windows are generated only within contiguous accepted-label segments.


## 9. Create 30-second windows with 15-second stride


Each sequence window has shape `(960, 6)`.


## 10. Map labels to stress/non-stress


Label 2 maps to `1`; labels 1 and 3 map to `0`.


## 11. Fit sequence scaler on training subjects only and save outputs


In [13]:
summary = run_preprocessing(PROJECT_ROOT)
split_shapes = summary["sequence_shapes"]
whole_data_shape = (sum(shape[0] for shape in split_shapes.values()), *next(iter(split_shapes.values()))[1:])
print("Whole windowed dataset shape (before split):", whole_data_shape)
print("Shapes after subject split:", split_shapes)
print("\nWindows per subject:")
windows_per_subject = summary.get("windows_per_subject")
if windows_per_subject is None:
    import pandas as pd
    window_metadata = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "metadata" / "windows_all.csv")
    windows_per_subject = window_metadata.groupby("subject_id", sort=False).size().to_dict()
for subject_id, window_count in windows_per_subject.items():
    print(f"{subject_id}: {window_count} windows")
summary


C:\Users\User\Documents\deep-learning\deep-learning\src\preprocessing.py:46: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(handle, encoding="latin1")


Whole windowed dataset shape (before split): (2142, 960, 6)
Shapes after subject split: {'train': (1429, 960, 6), 'validation': (287, 960, 6), 'test': (426, 960, 6)}

Windows per subject:
S3: 140 windows
S4: 140 windows
S6: 142 windows
S7: 142 windows
S8: 142 windows
S9: 142 windows
S10: 147 windows
S13: 144 windows
S16: 143 windows
S17: 147 windows
S5: 143 windows
S15: 144 windows
S2: 138 windows
S11: 144 windows
S14: 144 windows


{'sequence_shapes': {'train': (1429, 960, 6),
  'validation': (287, 960, 6),
  'test': (426, 960, 6)},
 'label_counts': {'train': {0.0: 999, 1.0: 430},
  'validation': {0.0: 201, 1.0: 86},
  'test': {0.0: 298, 1.0: 128}},
 'metadata_rows': {'train': 1429, 'validation': 287, 'test': 426}}

## 12. Verify saved artifacts


In [14]:
for path in [
    PROJECT_ROOT / "data" / "processed" / "sequence" / "X_train_raw.pt",
    PROJECT_ROOT / "data" / "processed" / "sequence" / "X_train.pt",
    PROJECT_ROOT / "data" / "processed" / "sequence" / "y_train.pt",
    PROJECT_ROOT / "data" / "processed" / "metadata" / "windows_all.csv",
    PROJECT_ROOT / "artifacts" / "preprocessing" / "sequence_channels.json",
    PROJECT_ROOT / "artifacts" / "preprocessing" / "sequence_scaler.joblib",
    PROJECT_ROOT / "artifacts" / "preprocessing" / "label_mapping.json",
    PROJECT_ROOT / "artifacts" / "preprocessing" / "class_distribution.csv",
]:
    print(path.relative_to(PROJECT_ROOT), path.exists())


data\processed\sequence\X_train_raw.pt True
data\processed\sequence\X_train.pt True
data\processed\sequence\y_train.pt True
data\processed\metadata\windows_all.csv True
artifacts\preprocessing\sequence_channels.json True
artifacts\preprocessing\sequence_scaler.joblib True
artifacts\preprocessing\label_mapping.json True
artifacts\preprocessing\class_distribution.csv True
